# 第15课 · 从结果倒推原因——高斯消元（Gaussian elimination）与 Ax=b 的三种命运

**学习目标**
1. 把线性方程组写成增广矩阵（augmented matrix） `[A|b]`，掌握矩阵方程 `Ax = b` 的矩阵形式
2. 掌握三种初等行变换（交换 ER1、倍乘 ER2、消元 ER3）并在具体例题中手动执行
3. 用 `rank(A)` 和 `rank([A|b])` 分类方程组的解：唯一解、无穷多解、无解
4. 实现 `classify_system(A, b)` 函数，返回 `'unique'`/`'many'`/`'none'`，通过三个补充例题验证
5. （收束时再挂）知道超定系统可用最小二乘 `lstsq`——Aurora 的 LPC 是真实场景之一

> **本课主交付**：`classify_system` + 三种命运。LPC / 最小二乘是动机与收尾，不要求你本课就推导滤波器。

**为什么对 Aurora 重要**：后面 LPC 会把声音建模成 `Ax=b`（x 是系数）；方程多于未知数时靠最小二乘。  
能否「唯一 / 无穷 / 无解」先取决于 `rank`——本课先把 rank 判据钉牢。

← **上一课**　[L14 · 特征值与 SVD](L14_eigen_svd.ipynb)

> 上节课学习了 **特征值与 SVD**：找到矩阵偏爱的方向，用 SVD 打开低秩世界。  
> 本课将探讨 **高斯消元**。

## 本课剧情：已知目的地，倒推出你走了哪条路

你知道某个线性变换的结果（`b`），也知道变换的规则（矩阵 `A`）。  
问题：原始输入 `x` 是什么？这就是 `Ax = b`。

现实中随处可见：
- （理想化例子）语音增强：已知录音 b 与某种线性模型 A，反推干净语音 x——后面信号处理会再遇到，今天先当 Ax=b 的故事
- 线性回归：已知数据 b，特征矩阵 A，找权重向量 x = argmin‖Ax-b‖
- 电路分析：已知电压 b，阻抗矩阵 A，求电流 x

方程组可能有三种命运：
- **唯一解**：A 满秩，`rank(A) = rank([A|b]) = n`
- **无数解**：A 列线性相关，消元后有自由变量
- **无解**：b 不在 A 的列空间中，方程矛盾

本课你将实现 `classify_system(A, b)` 来自动判断哪种情况。

## 插播：「秩」(rank) 和「列空间」(column space) 到底是什么？

在往下走之前，先停下来把两个会反复出现的词说清楚——**秩**和**列空间**。如果现在不搞懂，后面看到 "rank(A) < rank([A|b])" 这类判断会像看天书。

**先讲一个类比**：假设三个朋友分别给你指路去同一个地方。如果三条路线描述的其实是同一条路（只是换了个说法，比如"先左转再直走"和"直走之前先左转"），你实际上只拿到了 1 条"真正独立"的信息，即使听起来是 3 条。矩阵的**秩**做的就是这件事——数一数它的行（或列）里，究竟有几个是"真正独立"、能提供新信息的，重复的或能被别人拼出来的不算。

用数字说话：
- A = [[1,0],[0,1]]：两列 [1,0] 和 [0,1] 谁也拼不出谁，两条都独立 → rank(A) = 2。
- A = [[1,1],[1,1]]：两列都是 [1,1]，第二列 = 1×第一列，没有提供任何新信息 → rank(A) = 1，即使矩阵是 2×2 的。

**列空间是什么**：把 A 的所有列向量，任意加权（用任意实数系数）求和，你能落到的所有点，合起来就是 A 的"列空间"。可以想象成——你站在原点，每一列是一个能走的方向，你可以沿着这些方向走任意长度（正着走、反着走都行），也可以同时用几个方向合成一步走。所有你能走到的地方，就是列空间。就像列空间是一张你能到达的"纸面"（或一条"线"），目标点 b 如果不在这张纸上，你在纸上再怎么走也走不到它。

- 如果 A 只有一列 [1, 2]，列空间就是这一个方向能张成的所有点：`{t·[1,2] : t∈ℝ}`，也就是过原点、斜率为 2 的一条直线——你永远走不出这条线。
- 如果 A = [[1,1],[1,1]]（两列相同），无论 x₀、x₁ 怎么取，`x₀·[1,1] + x₁·[1,1] = (x₀+x₁)·[1,1]`——不管怎么组合，落点永远在"y=x"这条直线上。这里 rank(A)=1，列空间只是一条线，而不是整个平面。

现在就能回答"为什么 b 不在列空间就无解"：`Ax=b` 问的是"能不能用 A 的列拼出 b"。如果 b 根本不落在列空间划出的那条线（或那个平面）上，无论 x 怎么取都拼不出来——这就是无解。下面用代码验证一下：`b=[2,3]` 不在"y=x"线上（2≠3），拼不出来；`b=[2,2]` 在线上（2=2），能拼出来（而且因为这条线上只有 1 个独立方向、却有 2 个系数可调，拼法不止一种——这正是"无穷多解"的来源）。

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

A_dep = np.array([[1., 1.],
                   [1., 1.]])   # 两列相同 → rank=1，列空间只是一条直线 y=x

for label, b_test in [('b=[2,3] (不在 y=x 线上)', np.array([2., 3.])),
                       ('b=[2,2] (在 y=x 线上)',   np.array([2., 2.]))]:
    rank_A  = np.linalg.matrix_rank(A_dep)
    rank_Ab = np.linalg.matrix_rank(np.column_stack([A_dep, b_test]))
    print(f'{label}: rank(A)={rank_A}, rank([A|b])={rank_Ab}')

print()
print('第一种：rank(A)=1 < rank([A|b])=2 → b 引入了新方向 → 无解')
print('第二种：rank(A)=1 = rank([A|b])=1 → b 没有引入新方向，且只有1个独立方向却有2个系数 → 无穷多种拼法')


## 1. Ax=b：用列的线性组合拼出 b

看 `A @ x = b` 的"视角翻转"：

**列视角**：`A @ x = x[0]·col₀ + x[1]·col₁ + ... + x[n-1]·colₙ₋₁`  
→ 求解 `Ax=b` = 找一组系数 `x`，让 A 的各列加权组合精确等于 `b`

**例子**：A = [[2, 1], [0, 1]]，b = [5, 3]  
问：能用 x[0]×[2,0] + x[1]×[1,1] 拼出 [5,3] 吗？  
答案：x[0]=1, x[1]=3 → 1×[2,0] + 3×[1,1] = [2,0]+[3,3] = [5,3] ✓

**高斯消元的作用**：通过初等行变换，把增广矩阵 [A|b] 化成行阶梯形（REF），从最后一行"倒代入"（back substitution）逐个求解。

**rank 判断可解性**：
- `rank(A) = rank([A|b]) = n` → 唯一解
- `rank(A) = rank([A|b]) < n` → 无数解
- `rank(A) < rank([A|b])` → 无解

In [ ]:
import numpy as np

A = np.array([[2.0, 1.0],
              [0.0, 1.0]])
x = np.array([3.0, 2.0])
b = A @ x

print('A 的第 1 列:', A[:, 0])
print('A 的第 2 列:', A[:, 1])
print('x =', x)
print('b = 3 * col1 + 2 * col2 =', b)


## 1.1 再进入一个三元例题

下面的三元方程组只是同一件事的更大版本：`A` 有三列，`x` 有三个系数，目标是拼出右侧的 `b`。


$$2x + y - 2z = 10,\quad 3x + 2y + 2z = 1,\quad 5x + 4y + 3z = 4$$

课件答案：唯一解 $x=1,\,y=2,\,z=-3$。


## 符号入口：先看形状，再看运算

`A` 是 `(m, n)` 矩阵，`x` 是 `(n,)` 向量（vector），`A @ x` 得到 `(m,)` 向量。`m` 是方程个数，`n` 是未知数个数，这两个数的大小关系决定了方程组是超定（overdetermined）、恰定（square）还是欠定（underdetermined）。

In [ ]:
import numpy as np
A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10,1,4], float)
print('解 =', np.linalg.solve(A, b), ' (课件 [1, 2, -3])')

## 动手观察：先看 shape，再看数值

运行这段代码，注意 `A.shape`、`x.shape`、`b.shape` 三者的关系——`m`、`n` 的具体值决定了线性系统有多少方程、多少未知数，也决定了能否用 `np.linalg.solve` 求解。

In [ ]:
import numpy as np

# 利用 numpy 对不同秩矩阵观察秩的变化（解的分类留给习题 4 实现）
cases = [
    ('唯一解', np.array([[2.,1.],[0.,3.]]), np.array([5.,6.])),
    ('无穷多解', np.array([[1.,2.],[2.,4.]]), np.array([3.,6.])),
    ('无解', np.array([[1.,1.],[1.,1.]]), np.array([2.,3.])),
]
for label, A, b in cases:
    rank_A  = np.linalg.matrix_rank(A)
    rank_Ab = np.linalg.matrix_rank(np.column_stack([A, b]))
    n = A.shape[1]
    print(f'{label}: rank(A)={rank_A}, rank([A|b])={rank_Ab}, n={n}')
# 观察：三行数字对应三种情形，习题 4 就是把这些比较关系编码进 classify_system。


## 插播：往矩阵里加一列，秩为什么会变——对照上面三个例子逐一看

上面代码打出的三行数字，藏着"无解 vs 有解"的全部秘密。规律其实很朴素：往 A 里加一列 b，秩要么保持不变，要么恰好 +1，不会多也不会少。

为什么？把"秩"想成"列空间目前占了几维"。A 的列空间已经张出了一块区域（一条线、一个面……）。现在把 b 也加进来当作新的一列：
- 如果 b 本来就能被 A 的列线性组合拼出来（b 已经"住"在 A 的列空间里），那 b 不会带来任何新方向，秩不变。
- 如果 b 拼不出来，说明它指向了一个 A 的列从未覆盖过的全新方向，秩恰好 +1（多了这一个新方向，但也仅仅多了一个）。

对照上面三个例子：
- **唯一解**：A=[[2,1],[0,3]]，两列 [2,0] 和 [1,3] 方向不同，独立张满整个平面（rank(A)=2）。既然平面已经被占满，b=[5,6] 自然也能被拼出来，rank([A|b]) 还是 2——秩没有变大，因为 b 没能带来"新方向"。而且因为两个方向本来就独立，拼法唯一。
- **无穷多解**：A=[[1,2],[2,4]]，两列 [1,2] 和 [2,4] 其实是同一个方向（第二列=2×第一列），rank(A)=1，列空间只是一条线。b=[3,6] 恰好也在这条线上（[3,6]=3×[1,2]），所以 rank([A|b]) 仍是 1——没引入新方向。但这条线只有 1 个独立方向，却要用 2 个系数去凑，配比可以有无穷种（比如 x=(3,0) 或 x=(1,1) 都行，验证一下：1×[1,2]+1×[2,4]=[3,6] ✓）。
- **无解**：A=[[1,1],[1,1]]，rank(A)=1，列空间是"y=x"这条线。b=[2,3] 不在这条线上，它指向了一个全新方向，rank([A|b]) 变成 2——秩变大了，说明 b 用 A 的列怎么也拼不出来。

一句话记住：**rank([A|b]) 比 rank(A) 大，就是 b 把"新信息"带进来了，而这份新信息不是 A 的列能提供的——所以无解。**

## 代码实验：同一个 A，不同的 x，看输出如何线性变化

用几个不同的 `x` 向量计算 `A @ x`，确认矩阵乘法是线性映射——这是高斯消元能逐步消去未知数的前提。

In [ ]:
import numpy as np

# 消元后验证：Ax=b 的残差
A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10.,1.,4.])
x = np.linalg.solve(A, b)
residual = np.linalg.norm(A @ x - b)
print(f'解 x = {np.round(x,6)}')
print(f'残差 ||Ax-b|| = {residual:.2e}  （接近0=正确）')
print(f'验证：A@x = {np.round(A@x,6)}')


## 1.2 `np.linalg.solve` 给出答案，高斯消元解释过程

先用工具确认答案，再看增广矩阵如何一步步变形。这样后面的行变换有一个明确目标：把未知数逐个分离出来。


## 2. 初等行变换 ER1 / ER2 / ER3 与增广矩阵

- **ER1** 交换两行  `Ri ↔ Rj`
- **ER2** 某行乘非零常数  `Ri → k·Ri`
- **ER3** 某行加上另一行的 k 倍  `Ri → Ri + k·Rj`

下面手动做课件那一步：`R2→2R2−3R1`, `R3→2R3−5R1`，把第一列下方消成 0。


## 插播：为什么这三种行变换不会改变方程组的解？

这三条操作看起来像是在"随便折腾"矩阵，但每一条都有一个共同的承诺：**变换前后，能让方程组成立的 x 完全一样，一个都不多，一个都不少**。逐条看为什么。

**ER1（交换两行）**：两行交换，只是把"先看哪个方程、后看哪个方程"的顺序换了一下，两个方程本身一个字都没变。显然不会影响哪个 x 能同时满足它们。

**ER2（某行乘以非零常数 k）**：这一步相当于把一个等式的两边同时乘以同一个数。比如方程 `2x+y=5`，两边乘 3 变成 `6x+3y=15`——能让原方程成立的 (x,y)，代入新方程一样成立（两边都放大了 3 倍，等式关系没变）；反过来，新方程两边除以 3 就能还原成原方程，所以这一步是可逆的，解集完全不变（要求 k≠0，否则两边乘 0 会把方程变成 `0=0`，白白丢掉了原来的约束——这正是 ER2 强调"非零"的原因）。

**ER3（某行加上另一行的 k 倍）——最容易犯嘀咕的一条**：`R2 → R2 + k·R1` 翻译回方程语言：如果原来的方程 1 是 `a₁·x = c₁`，方程 2 是 `a₂·x = c₂`，新的"方程 2"就是把这两个方程加权相加：`(a₂+k·a₁)·x = c₂+k·c₁`。

关键问题：为什么这个新方程不会引入多余的约束，也不会丢掉原来的约束？

- 假设 x 同时满足原方程 1 和方程 2（即 `a₁·x=c₁` 且 `a₂·x=c₂`），把这两个等式分别乘 k 和 1 再相加：`a₂·x + k·a₁·x = c₂ + k·c₁`，恰好就是新方程——所以原来的解，新方程也满足。
- 反过来，如果 x 同时满足方程 1（没变）和新方程 2'，能不能推出它也满足旧方程 2？可以：用新方程 2' 减去 k 倍的方程 1，就精确地还原出旧方程 2。这一步是可逆的（"加回去"和"减回来"是一对逆操作），信息没有丢失。

所以 ER3 不是在"改变"方程组，而是在用旧方程"重新组合"出一个等价的说法——就像已知"甲的年龄"和"甲比乙大 3 岁"，可以推出"乙比甲小 3 岁"这类换个说法的推论，不会让原来关于甲乙年龄的答案变化。

下面用一个二元例子具体验证一遍：`x+y=3` 和 `2x-y=0`，先手解出真解，再做一次 ER3 变换，检查新方程组给出的解是否完全一样。

In [ ]:
import numpy as np

# 原方程组：x + y = 3,  2x - y = 0
eq1 = np.array([1., 1., 3.])   # [系数x, 系数y, 右端]
eq2 = np.array([2., -1., 0.])

A_old = np.array([eq1[:2], eq2[:2]])
b_old = np.array([eq1[2], eq2[2]])
x_old = np.linalg.solve(A_old, b_old)
print('原方程组的解 (x, y) =', x_old)

# ER3：eq2' = eq2 - 2*eq1  (选 k=-2，抵消掉 eq2 里 x 的系数 2)
eq2_new = eq2 - 2 * eq1
print('新方程 2：', eq2_new, ' → 对应方程', f'{eq2_new[0]:.0f}x + ({eq2_new[1]:.0f})y = {eq2_new[2]:.0f}')

A_new = np.array([eq1[:2], eq2_new[:2]])
b_new = np.array([eq1[2], eq2_new[2]])
x_new = np.linalg.solve(A_new, b_new)
print('变换后方程组的解 (x, y) =', x_new)

print('\n两次解出的 (x, y) 完全一致：', np.allclose(x_old, x_new))
print('新方程只是老方程的"重新组合"，没有引入新约束，也没有丢失旧约束。')


In [ ]:
A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10,1,4], float)
aug = np.column_stack([A, b])      # 增广矩阵 [A | b]
print('增广矩阵:\n', aug)
aug[1] = 2*aug[1] - 3*aug[0]       # ER3 + ER2
aug[2] = 2*aug[2] - 5*aug[0]
print('\n消元后(对照补充例题):\n', aug)


## 插播：消元时的倍数 2、3、5 是怎么选出来的？为什么不用分数？

代码里写的是 `aug[1] = 2*aug[1] - 3*aug[0]`，而不是看起来更直接的 `aug[1] = aug[1] - 1.5*aug[0]`。两种写法的目标是一回事：把第 2 行第 1 列的元素变成 0。

第 1 列现在是 `[2, 3, 5]`（分别来自第 1、2、3 行）。要把第 2 行的 3 消成 0：
- **方法 A（notebook 用的）**：`R2 → 2·R2 − 3·R1`。验证第 1 列：`2×3 − 3×2 = 0` ✓。这一步其实是 ER2（把 R2 整体乘 2）和 ER3（再减去 3 倍 R1）的组合——都是允许的初等行变换，合起来还是可逆、不改变解集的操作。
- **方法 B（更直接的单步 ER3）**：`R2 → R2 − 1.5·R1`。验证：`3 − 1.5×2 = 0` ✓，同样能消成 0，而且只用了一次操作。

两种方法都合法，化简到最后的解完全一样——毕竟不管用哪种系数，都没有跳出"初等行变换不改变解集"这条保证。选哪种纯粹是工程上的取舍：方法 A 全程只出现整数，手算、心算不容易出错，也没有累积舍入误差；方法 B 步骤更少，但会引入小数（在更大的矩阵里，小数可能一路累积精度损失）。`np.linalg.solve` 内部用的是更稳健的算法（LU 分解 + 主元选取），比这里手写的整数消元更进一步考虑了数值稳定性，但基本思路是一样的："每一步都朝着把某个位置消成 0 前进"。

下面代码用方法 B 在同一个方程组上重新算一遍，确认解仍然一致。

In [ ]:
import numpy as np

A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10,1,4], float)
aug_B = np.column_stack([A, b])

# 方法 B：单步 ER3，允许出现分数
aug_B[1] = aug_B[1] - 1.5*aug_B[0]
aug_B[2] = aug_B[2] - 2.5*aug_B[0]
print('方法 B 消元后:\n', aug_B)
print('\n对照方法 A（整数版）:\n [[2, 1, -2, 10], [0, 1, 10, -28], [0, 3, 16, -42]]')
print('第一列都变成了 [2, 0, 0]，第 2、3 行的比例关系一致（方法 B 是方法 A 整体除以 2）——两条路殊途同归。')


## 2.1 把消元做完：消到"行阶梯形"，再倒代入求解

上一步只处理了第 1 列（把它下方全部消成 0）。**行阶梯形**（row echelon form, REF）的目标是：每一行"打头"的非零数（叫主元，pivot）都严格排在上一行主元的右边，最终矩阵变成一个"楼梯"形状——左下角一大片全是 0。目前的矩阵是：

```
[2,  1, -2, 10]
[0,  1, 10, -28]
[0,  3, 16, -42]
```

还没到楼梯形——第 3 行的第 2 列还是 3，不是 0，需要继续消。用第 2 行（主元是 1，很好算）去消第 3 行：`R3 → R3 − 3·R2`。消完之后第 3 行就只剩最后一个未知数 z 了。

**倒代入（back substitution）** 的意思是：楼梯形矩阵最下面一行现在只含一个未知数，先解出它；解出来之后往上一行代入，这一行只剩一个未知数没解出，也能求出来；如此一路往上，直到求出第一个未知数。这就像剥洋葱——从最里面（约束最少的一层）剥起。

下面代码把这两步都做完，并且把倒代入的每一步都打印出来，跟 `np.linalg.solve` 给的答案对照。

In [ ]:
import numpy as np

A = np.array([[2,1,-2],[3,2,2],[5,4,3]], float)
b = np.array([10,1,4], float)
aug = np.column_stack([A, b])
aug[1] = 2*aug[1] - 3*aug[0]
aug[2] = 2*aug[2] - 5*aug[0]
print('消完第1列:\n', aug)

# 继续消元：把第3行第2列也消成0，得到真正的行阶梯形
aug[2] = aug[2] - 3*aug[1]
print('\n消完第2列 → 行阶梯形 (REF):\n', aug)

# 倒代入：从最后一行开始，逐个解出 z, y, x
row3 = aug[2]                       # [0, 0, c, d]  → c*z = d
z = row3[3] / row3[2]
print(f'\n第3行 {row3[2]:.0f}·z = {row3[3]:.0f}  →  z = {z:.4f}')

row2 = aug[1]                       # [0, b, c, d]  → b*y + c*z = d
y = (row2[3] - row2[2]*z) / row2[1]
print(f'第2行 {row2[1]:.0f}·y + {row2[2]:.0f}·z = {row2[3]:.0f}  →  y = {y:.4f}')

row1 = aug[0]                       # [a, b, c, d]  → a*x + b*y + c*z = d
x = (row1[3] - row1[1]*y - row1[2]*z) / row1[0]
print(f'第1行 {row1[0]:.0f}·x + {row1[1]:.0f}·y + {row1[2]:.0f}·z = {row1[3]:.0f}  →  x = {x:.4f}')

print(f'\n手算倒代入结果: x={x:.4f}, y={y:.4f}, z={z:.4f}')
print('np.linalg.solve 结果:', np.linalg.solve(A, b))
print('两者一致 ✅ —— 消元 + 倒代入，就是 solve 内部在做的事（只是它用了更稳健的算法）。')


## 3. 三种情形：用秩(rank)判断

比较系数矩阵 A 的秩与增广矩阵 [A|b] 的秩、未知数个数 n：
- `rank(A) = rank([A|b]) = n` → **唯一解**
- `rank(A) = rank([A|b]) < n` → **无穷多解**
- `rank(A) < rank([A|b])` → **无解**（矛盾）


## 4. ✏️ 实现 `classify_system(A, b)`

**推理路线**：
1. 把 A 和 b 拼成增广矩阵 [A|b]，分别计算 `rA = rank(A)` 和 `rAb = rank([A|b])`。
2. 若 `rA < rAb`：b 不在 A 的列空间里，方程矛盾，返回 `'none'`。
3. 若 `rA == rAb == n`：每个未知数对应一个独立约束，返回 `'unique'`。
4. 若 `rA == rAb < n`：约束数不够，存在自由变量，返回 `'many'`。

**参考输入输出**：
- `A=[[2,1],[1,3]], b=[5,10]` → `'unique'`（rank=2=n）
- `A=[[1,2],[2,4]], b=[3,6]` → `'many'`（rank=1<n=2，b 在列空间里）
- `A=[[1,2],[2,4]], b=[3,7]` → `'none'`（rank(A)=1 < rank([A|b])=2）

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


写 `classify_system` 前明确三件事：
- 输入：`A`（系数矩阵，`(m, n)`）、`b`（右端向量，`(m,)`）
- 关键步骤：计算 `rA = rank(A)`，`rAb = rank([A|b])`，再与 `n = A.shape[1]` 比较
- 返回：字符串 `'unique'`、`'many'` 或 `'none'`

In [ ]:
def classify_system(A, b):
    A = np.asarray(A, float); b = np.asarray(b, float)
    # ✏️ TODO: 用秩判断解的情形，返回 'unique'/'many'/'none'
    raise NotImplementedError(
        "TODO: implement classify_system → return 'unique'/'many'/'none'"
    )


In [ ]:
# 三个补充例题 (3×3) + 超定例题 (4×2，对应 Aurora LPC 超定场景)
U  = (np.array([[2,1,-2],[3,2,2],[5,4,3]],float), np.array([10,1,4],float))
M  = (np.array([[1,2,-3],[2,-1,4],[4,3,-2]],float), np.array([6,2,14],float))
N  = (np.array([[1,2,-3],[3,-1,2],[5,3,-4]],float), np.array([-1,7,2],float))
# 超定 (4 方程, 2 未知数)：rank=2=n，b 在列空间内 → 'unique'
OD = (np.array([[1.,0.],[0.,1.],[1.,1.],[2.,1.]]), np.array([1.,1.,2.,3.]))

try:
    print('例1 (unique):', classify_system(*U),  '  预期: unique')
    print('例2 (many):  ', classify_system(*M),  '  预期: many')
    print('例3 (none):  ', classify_system(*N),  '  预期: none')
    print('超定 (unique):', classify_system(*OD), '  预期: unique (最小二乘意义)')
    assert classify_system(*U)  == 'unique', '例1 失败'
    assert classify_system(*M)  == 'many',   '例2 失败'
    assert classify_system(*N)  == 'none',   '例3 失败'
    assert classify_system(*OD) == 'unique', '超定例题失败'
    print('\n✅ 通过：三个补充例题 + 超定例题的解类型检查完成。')
except (NotImplementedError, TypeError) as e:
    print(f'⚠️  任务未完成：{e}')
    print('请先实现上方 classify_system 函数再运行此单元格。')


### 参考对照：`classify_system` 核心逻辑（完成习题后再展开）

> 以下是可运行的参考实现，供完成习题后核对。未完成前请勿提前查看。


In [ ]:
# ── 参考实现（仅供完成习题后对照）──
def _classify_system_ref(A, b):
    A  = np.asarray(A, float); b = np.asarray(b, float)
    rA  = np.linalg.matrix_rank(A)
    rAb = np.linalg.matrix_rank(np.column_stack([A, b]))
    n   = A.shape[1]
    if rA < rAb:
        return 'none'
    elif rA == rAb == n:
        return 'unique'
    else:
        return 'many'

# 验证参考实现本身正确
assert _classify_system_ref(*U)  == 'unique'
assert _classify_system_ref(*M)  == 'many'
assert _classify_system_ref(*N)  == 'none'
assert _classify_system_ref(*OD) == 'unique'
print('参考实现自检通过 ✅')


**🔗 Aurora 连接**：线性回归的正规方程 `(XᵀX)w = Xᵀy` 就是一个线性系统；可解性对应模型何时有唯一最优解。

**补充例题对应**：增广矩阵、阶梯形、初等行变换、解的分类。


## 插播：为什么 4 个方程、2 个未知数，还能有"唯一解"？

直觉上，方程越多、未知数越少，好像应该越难满足——要么互相打架（无解），要么至少不会更"唯一"。但前面的超定例题 `OD` 却打印出 `unique`，秘密藏在 `rank(A)=2=n` 这一行。

先看这四个方程到底在说什么。`OD` 的 `A` 和 `b` 是：

```
[1, 0]        [1]
[0, 1]   x =  [1]
[1, 1]        [2]
[2, 1]        [3]
```

翻译成人话：`x=1`（第1行）、`y=1`（第2行）、`x+y=2`（第3行）、`2x+y=3`（第4行）。前两行已经把 x 和 y 都钉死了——不需要更多信息。第 3、4 行看起来是"额外"的方程，但代入 x=1, y=1 一验证：`x+y=1+1=2` ✓，`2x+y=2+1=3` ✓——它们不多不少，恰好和前两行给出的答案吻合，没有添加任何新约束，也没有产生任何矛盾。

用秩的语言说：第 3 行其实等于"第 1 行 + 第 2 行"，第 4 行等于"2×第 1 行 + 第 2 行"——它们是前两行的**线性组合**，没有指向任何新方向，所以 `rank(A)` 仍然只有 2（不是 4）。同时因为 b 里对应的第 3、4 个值也恰好满足同样的线性组合关系（`2=1+1`，`3=2×1+1`），增广矩阵的秩也还是 2——没有出现"多一个方向、b 却不跟着走"的矛盾情况。`rank(A)=rank([A|b])=n=2`，正好落进"唯一解"的判定条件。

**这是一个特意设计的"温顺"超定系统**。更常见的超定场景（比如 Aurora 的 LPC：几百个采样点的方程，只有十几个滤波器系数）里，多出来的方程通常*不会*恰好一致——实际测得的方程会因为噪声互相有一点点矛盾，严格意义上"无解"（`rank(A) < rank([A|b])`）。这时候"唯一解"就要重新定义："最小二乘意义下的唯一解"，指的不是"精确满足所有方程的 x"（不存在），而是"让所有方程的误差平方和 `‖Ax-b‖²` 最小的那个 x"——这个 x 通常是唯一确定的，靠的正是前面讲过的 `lstsq`。

下面代码验证一下"第 3、4 行是前两行线性组合"这件事，并且额外造一个"不温顺"的超定例子，看 `lstsq` 在真正矛盾的超定系统里怎么办。

In [ ]:
import numpy as np

A_od, b_od = OD
print('第3行 = 第1行+第2行？', np.allclose(A_od[2], A_od[0] + A_od[1]),
      ' b[2]=1+1?', b_od[2] == b_od[0] + b_od[1])
print('第4行 = 2*第1行+第2行？', np.allclose(A_od[3], 2*A_od[0] + A_od[1]),
      ' b[3]=2+1?', b_od[3] == 2*b_od[0] + b_od[1])
print(f'rank(A) = {np.linalg.matrix_rank(A_od)}  (不是4，因为后两行没有新方向)')

print()
print('--- 对照：一个"不温顺"的超定系统（4个方程，2个未知数，但有噪声、互相矛盾）---')
rng = np.random.default_rng(0)
A_noisy = np.array([[1.,0.],[0.,1.],[1.,1.],[2.,1.]])
b_noisy = np.array([1., 1., 2., 3.]) + rng.normal(0, 0.05, size=4)  # 加一点点噪声
rank_noisy    = np.linalg.matrix_rank(A_noisy)
rank_noisy_ab = np.linalg.matrix_rank(np.column_stack([A_noisy, b_noisy]))
print(f'rank(A)={rank_noisy}, rank([A|b])={rank_noisy_ab}  → 严格来说已经是 "none"（无精确解）')
x_best, res, rank_, sv_ = np.linalg.lstsq(A_noisy, b_noisy, rcond=None)
print(f'lstsq 给出的"最小二乘意义唯一解": x={np.round(x_best,4)}，残差={np.linalg.norm(A_noisy@x_best-b_noisy):.4f}')
print('这正是 Aurora LPC 系数估计的真实处境：方程比未知数多，噪声让它们不完全一致，')
print('但 lstsq 依然能给出一个明确、可复现、误差最小的答案。')


## 🎨 图示：Ax=b 看成 A 各列的线性组合 (补充例题 1)


In [ ]:
from aurora.laviz import style, mat_times_vec
style()
mat_times_vec([[2,1,-2],[3,2,2],[5,4,3]],[1,2,-3]);  # = b=[10,1,4]

## 插播：条件数(condition number)到底在测量什么？

先想一个日常场景：用一把很钝的剪刀剪纸，手稍微抖一下，剪出来的线就歪很多；换一把锋利、稳定的剪刀，同样的手抖，剪出来的线几乎不受影响。**条件数**衡量的就是矩阵这台"机器"对"输入抖动"有多敏感——数值上，`κ(A)` 越大，`b` 里一点点误差（哪怕只是浮点数舍入产生的），都会被求解过程放大成 `x` 里一个大得多的误差。

为什么会这样？回到"列空间"的画面：如果 A 的两行（或两列）几乎指向同一个方向（比如下面例子里的 `[1,1]` 和 `[1,1+eps]`，eps 很小时几乎重合），那么这两个方程提供的信息几乎是"同一句话说了两遍"，能真正把 x 锁定住的"独立信息"很少——一点点扰动就可能让"交点"跑到很远的地方（想象两条几乎平行的线，稍微转一点点角度，交点就会跳到很远处；如果两条线夹角很大，稍微转一下交点几乎不动）。`κ(A)` 越大，就是在说"这两行越接近平行"，方程组对噪声越敏感。

条件数的正式定义是 `κ(A) = ‖A‖·‖A⁻¹‖`（矩阵范数乘上它的逆的范数），也可以用 L14 学过的奇异值算：`κ(A) = σ_max / σ_min`（最大奇异值除以最小奇异值）。`κ(A)=1` 是最理想的情况（比如单位矩阵，完全不放大误差）；`κ(A)` 越大于 1，误差被放大得越厉害；`κ(A) → ∞` 就是矩阵变得奇异（下面马上会讲）。

光说不练不够直观——下面先重复原来的实验（观察 κ(A) 如何随 eps 变化），紧接着补一个关键演示：**同样给 b 加一点点噪声，κ(A) 小的矩阵 x 几乎不变，κ(A) 大的矩阵 x 会跳得很远**。

In [ ]:
import numpy as np

# 参数实验：A 的行向量越接近线性相关，条件数爆炸
for eps in [0.1, 0.01, 0.001, 1e-6]:
    A = np.array([[1., 1.],[1., 1.+eps]])
    kappa = np.linalg.cond(A)
    print(f'eps={eps:.0e}  κ(A)={kappa:.2e}')
print('→ eps→0 时行向量几乎平行，Ax=b 数值求解越来越不稳定。')


In [ ]:
import numpy as np

# 对比：条件数很小 vs 条件数很大的矩阵，同样一点点 b 的扰动，x 的变化差多少
A_good = np.array([[1., 0.], [0., 1.]])          # 单位矩阵，κ=1，最稳
A_bad  = np.array([[1., 1.], [1., 1.0001]])       # 两行几乎平行，κ 很大

b = np.array([2., 2.])
noise = np.array([0.001, -0.001])                # 同样大小的一点点扰动

for label, Amat in [('κ(A)≈1 (良态)', A_good), (f'κ(A)≈{np.linalg.cond(A_bad):.0e} (病态)', A_bad)]:
    x0 = np.linalg.solve(Amat, b)
    x1 = np.linalg.solve(Amat, b + noise)
    shift = np.linalg.norm(x1 - x0)
    print(f'{label}: b 扰动 0.001 时，x 的变化量 = {shift:.4f}')

print('\n结论：b 的扰动完全一样大，但病态矩阵（κ 很大）把误差放大了几个数量级——')
print('这就是"条件数大 → 数值不稳定"的真实含义：不是求不出解，而是求出的解可能完全不可信。')


## 插播：行列式(determinant)是什么？"奇异矩阵"又是什么意思？

**行列式**是一个把方阵"压缩"成一个数字的运算，这个数字告诉你：这个矩阵对应的线性变换，把空间的面积（或体积）放大了多少倍，以及有没有翻转方向。

- 对 2×2 矩阵 `A=[[a,b],[c,d]]`，行列式定义为 `det(A) = a·d − b·c`。几何上，这就是 A 的两列向量 `[a,c]` 和 `[b,d]` 张成的平行四边形的面积（带符号——如果两个方向被"翻转"了，面积算作负的）。
- 对 3×3 矩阵，行列式对应的是三个列向量张成的平行六面体的体积，公式更长，但意思一样。

**为什么 `det(A)=0` 就意味着"奇异"（singular）**：如果行列式是 0，说明这个"平行四边形/六面体"被压扁了——面积或体积变成了 0。也就是说，A 的列向量们不再各自独立张出一片"新"区域，而是挤在了一条线（甚至一个点）上——这正是"列线性相关"、`rank(A) < n` 的几何画面。矩阵把原本的空间压扁成了一个更低维的东西，这个压扁的过程没法"反着走回去"（把一张压扁的纸复原成正方体是不可能的），所以 A 没有逆矩阵，`np.linalg.solve` 找不到唯一解，只能报错 `LinAlgError: Singular matrix`。

举例：`A=[[1,2],[2,4]]`，`det(A) = 1×4 − 2×2 = 0`——果然是奇异矩阵，这也正是本课"无穷多解/无解"例题反复用到的那个矩阵：它的两列 `[1,2]` 和 `[2,4]` 方向相同（后者是前者的 2 倍），张不出一个平面，只能张出一条线，验证了前面"列空间退化成一条线"的说法。

下面代码验证一下这个 2×2 行列式公式。

In [ ]:
import numpy as np

A_demo = np.array([[1., 2.], [2., 4.]])
a, b_, c, d = A_demo[0,0], A_demo[0,1], A_demo[1,0], A_demo[1,1]
det_manual = a*d - b_*c
det_numpy  = np.linalg.det(A_demo)
print(f'手算 det(A) = a*d - b*c = {a}*{d} - {b_}*{c} = {det_manual}')
print(f'np.linalg.det(A) = {det_numpy:.10f}  (浮点误差下约等于0)')
print('→ det(A)=0，两列共线，张不出平面 → 奇异矩阵，没有逆。')


## 参数实验：奇异矩阵如何触发错误，lstsq 如何补救

把 A 的行列式设为 0（如 `A=[[1,2],[2,4]]`），调用 `np.linalg.solve(A, b)` 会触发 `LinAlgError: Singular matrix`——奇异矩阵没有逆，无法得到唯一解。再用 `np.linalg.lstsq(A, b, rcond=None)` 处理同一个矩阵，它不报错，返回最小范数解（有无穷解时）或残差最小的近似解（方程矛盾时）。每次只改一个量，先预测 `solve` 还是 `lstsq` 会成功，再运行验证。

In [ ]:
import numpy as np

A_sing = np.array([[1., 2.],
                   [2., 4.]])   # 行列式 = 0，奇异矩阵

# ── 场景 1：一致方程（b 在列空间内） ──
b_consistent   = np.array([3., 6.])  # [3,6] = 3*[1,2]
b_inconsistent = np.array([3., 7.])  # [3,7] 不在 [1,2] 张成的空间里

print('=== np.linalg.solve（奇异矩阵应抛出 LinAlgError）===')
for label, bv in [('一致', b_consistent), ('矛盾', b_inconsistent)]:
    try:
        x = np.linalg.solve(A_sing, bv)
        print(f'  {label} b: 意外成功，x={x}')
    except np.linalg.LinAlgError as e:
        print(f'  {label} b: ✅ 捕获 LinAlgError — {e}')

print()
print('=== np.linalg.lstsq（不报错，返回最小范数/最小残差解）===')
for label, bv in [('一致', b_consistent), ('矛盾', b_inconsistent)]:
    x_ls, res, rank, sv = np.linalg.lstsq(A_sing, bv, rcond=None)
    residual = np.linalg.norm(A_sing @ x_ls - bv)
    print(f'  {label} b: x_ls={np.round(x_ls,4)},  残差={residual:.4f},  rank={rank}')
print()
print('结论：solve 在奇异矩阵时失败；lstsq 始终给出答案，')
print('      但一致方程残差≈0，矛盾方程残差>0——用 classify_system 可提前判断。')


## 插播：`lstsq` 返回的 4 个值，以及"最小范数解"/"最小残差解"到底在挑哪个答案

代码里写的 `x_ls, res, rank, sv = np.linalg.lstsq(A, b, rcond=None)`，这四个名字各自的意思：

1. **`x_ls`**：最终给出的解向量——这是 4 个返回值里唯一你通常会用到的。
2. **`res`**：残差平方和 `‖Ax−b‖²`（只有在方程"超定且无精确解"时才会算出非空数组；方程有精确解或欠定时，`res` 经常是空数组）。
3. **`rank`**：`lstsq` 内部自己算出来的 `rank(A)`，可以拿来跟你自己算的 `np.linalg.matrix_rank(A)` 对照，正常情况下两者应该一致。
4. **`sv`**：A 的**奇异值**（singular values，L14 讲过的 SVD 里的那组数）。奇异值的个数永远等于 A 的列数，但**非零奇异值的个数就等于 rank(A)**——如果某个奇异值几乎是 0（比如 `1.8e-16`，只是浮点误差意义上的 0），说明对应的那个方向"名存实亡"，不提供真正独立的信息，这也是为什么"数非零奇异值"是计算 rank 的一种实用方法。

现在解释两种"解"的选法，它们背后都是一种"公平的默认选择"：

- **无穷多解时 → 最小范数解**：当方程有无穷多个解时（比如前面 `x=(3,0)` 和 `x=(1,1)` 都满足 `[1,2]x₀+[2,4]x₁=[3,6]`），`lstsq` 不会随便挑一个，而是在所有解里，挑**范数（也就是 L11 学过的向量长度）最小**的那个——可以理解成"在不违反约束的前提下，选一个最省力、最不夸张的答案"，就像有无数种走法能把购物车推到收银台，`lstsq` 默认选那条走得最近的路。
- **无解时 → 最小残差解**：当方程无解时（b 不在列空间里，怎么选 x 都无法让 `Ax` 精确等于 b），`lstsq` 转而去找一个 x，让 `‖Ax−b‖`（"猜的答案"和"真实目标"之间的距离）尽可能小——这就是"残差最小"。这个 x 是唯一确定的（只要 A 列满秩），因为"离 b 最近的、落在列空间里的点"是唯一的一个投影点（想象从 b 这个点向列空间这片"墙"作垂线，垂足只有一个）。

下面代码把 `sv`（奇异值）和 `res`（残差）都打印出来，对照"一致方程"和"矛盾方程"两种情况看它们的差别。

In [ ]:
import numpy as np

A_sing = np.array([[1., 2.], [2., 4.]])
b_consistent   = np.array([3., 6.])
b_inconsistent = np.array([3., 7.])

for label, bv in [('一致 (b=[3,6])', b_consistent), ('矛盾 (b=[3,7])', b_inconsistent)]:
    x_ls, res, rank, sv = np.linalg.lstsq(A_sing, bv, rcond=None)
    print(f'--- {label} ---')
    print(f'  x_ls = {np.round(x_ls, 4)}   (最小范数解 / 最小残差解)')
    print(f'  res  = {res}   (只有"矛盾且超定"时才非空)')
    print(f'  rank = {rank}   (对照 np.linalg.matrix_rank = {np.linalg.matrix_rank(A_sing)})')
    print(f'  sv   = {np.round(sv, 6)}   (奇异值；非零个数 = rank)')
    print(f'  手算残差 ‖Ax-b‖ = {np.linalg.norm(A_sing @ x_ls - bv):.6f}')
    print()

print('sv 里一个约 5.0，一个几乎是 0 —— 只有 1 个"真正有效"的方向，对应 rank(A)=1。')


## 插播：到底该用 `solve` 还是 `lstsq`？

一条简单的决策流程，串联本课学的所有内容：

1. 先跑 `classify_system(A, b)`（或者直接比较 `rank(A)` 与 `rank([A|b])`、`n`）。
2. 如果结果是 `'unique'`：`np.linalg.solve(A, b)` 直接可用，会给出唯一精确解。
3. 如果结果是 `'many'`：`solve` 通常会报错或结果不稳定（它假设唯一解存在），改用 `np.linalg.lstsq(A, b)`——它会在无穷多个解里给出"最小范数"的那个，是一个有明确规则、可复现的默认答案。
4. 如果结果是 `'none'`：`solve` 一定报错或给出错误答案，`lstsq` 是唯一选择——它给出的是"最接近"的近似解（最小残差解），这也是 Aurora 里 LPC 系数估计实际使用最小二乘法的原因：真实录音的自相关矩阵几乎不可能恰好方阵满秩，最小二乘法能在"没有精确解"的情况下仍然给出一个有意义、可用的答案。

## 本课收束

现在可以用 `np.linalg.solve(A, b)` 求唯一解，用 `classify_system(A, b)` 判断方程组的可解性。Aurora 的 LPC 系数估计把录音片段的自相关矩阵当作 `A`，解这个线性系统得到预测滤波器 `x`；`rank(A)` 不足时需要切换到最小二乘形式。下一节介绍行列式与逆矩阵，给出 `rank` 降低时系统退化的几何解释。

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：线性方程组手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：方程组 `2x + y = 5, x + 3y = 8`  
写出增广矩阵 `[A|b]`，消元后求 x 和 y。

**问 2**：A = [[1, 2], [2, 4]]，b = [1, 3]，  
这个方程组有唯一解、无数解还是无解？（手算 rank(A) 和 rank([A|b])）

**问 3**：A = [[1, 2], [2, 4]]，b = [2, 4]，  
和问 2 的矩阵完全一样，但 b 变了——现在是哪种情况？

**问 4**：线性方程组 `Ax=b` 中，若 A 是 4×4 矩阵且 `rank(A) = 4`，  
则对任意 b，方程组有几个解？

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：2x+y=5, x+3y=8
A1 = np.array([[2., 1.], [1., 3.]])
b1 = np.array([5., 8.])
x1 = np.linalg.solve(A1, b1)
assert np.allclose(A1 @ x1, b1, atol=1e-10)
print(f"Q1 ✅  2x+y=5, x+3y=8 → x={x1[0]:.4f}, y={x1[1]:.4f}")
print(f"       验证 A@x = {np.round(A1 @ x1, 4)} = b = {b1}")

# 问2：rank 判断（A行线性相关，b矛盾）
A2 = np.array([[1., 2.], [2., 4.]])
b2_no = np.array([1., 3.])
rA2 = np.linalg.matrix_rank(A2)
rAb2 = np.linalg.matrix_rank(np.column_stack([A2, b2_no]))
print(f"\nQ2 ✅  rank(A)={rA2}, rank([A|b])={rAb2} → 无解（rank 不等，b 不在列空间）")
try:
    cs = classify_system(A2, b2_no)
    assert 'no' in cs.lower() or '无' in cs or 'inconsistent' in cs.lower()
    print(f"       classify_system → '{cs}'")
except (NotImplementedError, TypeError):
    print(f"       (classify_system 待实现)")

# 问3：同A，b变为[2,4]（在列空间）
b2_inf = np.array([2., 4.])
rAb3 = np.linalg.matrix_rank(np.column_stack([A2, b2_inf]))
print(f"\nQ3 ✅  rank(A)={rA2}, rank([A|b])={rAb3}, n=2 → rank(A)<n → 无数解")
try:
    cs3 = classify_system(A2, b2_inf)
    assert 'inf' in cs3.lower() or '无数' in cs3 or 'many' in cs3.lower()
    print(f"       classify_system → '{cs3}'")
except (NotImplementedError, TypeError):
    print(f"       (classify_system 待实现)")

# 问4：4×4满秩 → 唯一解
A4 = np.eye(4) * 2 + np.random.default_rng(0).standard_normal((4,4)) * 0.1
rank_A4 = np.linalg.matrix_rank(A4)
assert rank_A4 == 4, f"矩阵秩应为4"
print(f"\nQ4 ✅  4×4 满秩矩阵（rank={rank_A4}=n=4）→ 唯一解（对任意 b）")
print("\n🎉 线性方程组白板挑战通过！rank 判断可解性已内化。")

In [ ]:
# ✏️ 本课自评
l15_review = {
    "classify_system_implemented": None,  # classify_system 实现并通过断言？True/False
    "gaussian_elimination_understood": None,  # 理解增广矩阵消元过程？True/False
    "rank_solvability_rule":       None,  # 能用 rank 判断唯一/无数/无解？True/False
    "column_space_intuition":      None,  # 理解"b 在 A 的列空间"的含义？True/False
    "whiteboard_passed":           None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l15_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l15_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L15 全部通关！进入 L16：行列式与逆矩阵')

---

→ **下一课**　[L16 · 行列式与逆矩阵](L16_determinant_inverse.ipynb)

> 下节课将学习 **行列式与逆矩阵**：det(A) 的几何含义（面积缩放），求逆与何时不可逆。